In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import linear_model
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
# load data dari file CSV
data_test = pd.read_csv('data/test.csv')
data_test.head()

In [ ]:
# menampilkan informasi dataset
data_test.info()

In [ ]:
# Menampilkan statistik deskriptif dari dataset
data_test.describe(include="all")
'''
count -> jumlah data yang tidak kosong
unique -> jumlah nilai yang unik(hanya untuk kolom kategori)
top -> nilai yang paling sering muncul(hanya untuk kolom kategori)
freq -> frekuensi kemunculan nilai top(hanya untuk kolom kategori)
mean -> rata-rata nilai(numerik)
std -> standar deviasi(numerik) Artinya:
    Seberapa menyebar datanya
    kecil → data mirip-mirip
    besar → data bervariasi
min -> nilai minimum(numerik)
25% -> nilai pada persentil ke-25(numerik)
50% -> nilai pada persentil ke-50(numerik)
75% -> nilai pada persentil ke-75(numerik)
max -> nilai maksimum(numerik)
'''

In [ ]:
# check missing values
missing_values = data_test.isnull().sum()
print(missing_values)
missing_values[missing_values > 0]

Remove Missing Values

Pertama-tama, Pisahkan kolom yang memiliki missing value lebih dari 75% dan kurang dari 75%.

In [ ]:
# remove missing values
less = missing_values[missing_values < 1000].index
over = missing_values[missing_values >= 1000].index
print("Columns less than 75%:", less)
print("Columns more than 75%:", over)

Mengisi Nilai yang Hilang: data di atas terdapat beberapa fitur yang memiliki missing value kurang dari 75% dari jumlah skala pada data. Namun, perlu Anda catat bahwa seluruh fitur tersebut memiliki tipe data yang berbeda. Sehingga penanganan missing value-nya pun perlu dibedakan.

In [ ]:

# Contoh mengisi nilai yang hilang dengan median untuk kolom numerik
numeric_features = data_test.select_dtypes(include=['number']).columns
print("Numeric features:", numeric_features)
data_test[numeric_features] = data_test[numeric_features].fillna(data_test[numeric_features].median())

In [ ]:
# Contoh mengisi nilai yang hilang dengan mode untuk kolom kategori
kategorical_features = data_test[less].select_dtypes(include=['object']).columns
 
for column in kategorical_features:
    data_test[column] = data_test[column].fillna(data_test[column].mode()[0])

In [ ]:
# drop columns with more than 75% missing values
df = data_test.drop(columns=over)

# final check missing values
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

In [ ]:
for feature in numeric_features:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[feature])
    plt.title(f'Box Plot of {feature}')
    plt.show()

In [ ]:
# Contoh sederhana untuk mengidentifikasi outliers menggunakan IQR
Q1 = df[numeric_features].quantile(0.25)
Q3 = df[numeric_features].quantile(0.75)
IQR = Q3 - Q1
print(IQR)

Menghapus outlier

In [ ]:
# Filter dataframe untuk hanya menyimpan baris yang tidak mengandung outliers pada kolom numerik
condition = ~((df[numeric_features] < (Q1 - 1.5 * IQR)) | (df[numeric_features] > (Q3 + 1.5 * IQR))).any(axis=1)
print(condition)
df_filtered_numeric = df.loc[condition, numeric_features]
 
# Menggabungkan kembali dengan kolom kategorikal
categorical_features = df.select_dtypes(include=['object']).columns
df = pd.concat([df_filtered_numeric, df.loc[condition, categorical_features]], axis=1)

Series([], dtype: bool)


In [ ]:
# Standardisasi fitur numerik
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

In [ ]:
# Histogram Sebelum Standardisasi
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(data_test[numeric_features[3]], kde=True)
plt.title("Histogram Sebelum Standardisasi")

In [ ]:
# Histogram Setelah Standardisasi
plt.subplot(1, 2, 2)
sns.histplot(df[numeric_features[3]], kde=True)
plt.title("Histogram Setelah Standardisasi")

In [ ]:
# Mengidentifikasi baris duplikat
duplicates = df.duplicated()
 
print("Baris duplikat:")
print(df[duplicates])

In [ ]:
# Menghapus baris duplikat
df = df.drop_duplicates()
 
print("DataFrame setelah menghapus duplikat:")
print(df)

ENCODING

In [ ]:
category_features = df.select_dtypes(include=['object']).columns
df[category_features]

In [ ]:
df_one_hot = pd.get_dummies(df, columns=category_features)
df_one_hot

In [ ]:
# Inisialisasi LabelEncoder
label_encoder = LabelEncoder()
df_lencoder = pd.DataFrame(df)
 
for col in category_features:
    df_lencoder[col] = label_encoder.fit_transform(df[col])
 
# Menampilkan hasil
df_lencoder

In [ ]:
# Menghitung jumlah dan persentase missing values di setiap kolom
missing_values = df_lencoder.isnull().sum()
missing_percentage = (missing_values / len(df_lencoder)) * 100
 
missing_data = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
}).sort_values(by='Missing Values', ascending=False)
 
missing_data[missing_data['Missing Values'] > 0] 

In [ ]:
# Menghitung jumlah variabel
num_vars = df_lencoder.shape[1]
 
# Menentukan jumlah baris dan kolom untuk grid subplot
n_cols = 4  # Jumlah kolom yang diinginkan
n_rows = -(-num_vars // n_cols)  # Ceiling division untuk menentukan jumlah baris
 
# Membuat subplot
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
 
# Flatten axes array untuk memudahkan iterasi jika diperlukan
axes = axes.flatten()
 
# Plot setiap variabel
for i, column in enumerate(df_lencoder.columns):
    df_lencoder[column].hist(ax=axes[i], bins=20, edgecolor='black')
    axes[i].set_title(column)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
 
# Menghapus subplot yang tidak terpakai (jika ada)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
 
# Menyesuaikan layout agar lebih rapi
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi distribusi data untuk beberapa kolom
columns_to_plot = ['OverallQual', 'YearBuilt', 'LotArea', 'SaleType', 'SaleCondition']
 
plt.figure(figsize=(15, 10))
for i, column in enumerate(columns_to_plot, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df_lencoder[column], kde=True, bins=30)
    plt.title(f'Distribution of {column}')
 
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi korelasi antar variabel numerik
plt.figure(figsize=(12, 10))
correlation_matrix = df_lencoder.corr()
 
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Menghitung korelasi antara variabel target dan semua variabel lainnya
target_corr = df_lencoder.corr()['SaleType']
 
# (Opsional) Mengurutkan hasil korelasi berdasarkan korelasi
target_corr_sorted = target_corr.abs().sort_values(ascending=False)
 
plt.figure(figsize=(20, 6))
target_corr_sorted.plot(kind='bar')
plt.title(f'Correlation with SaleType')
plt.xlabel('Variables')
plt.ylabel('Correlation Coefficient')
plt.show()

**DATA SPLIT**

In [ ]:
# Memisahkan fitur (X) dan target (y)
X = df_lencoder.drop(columns=['SaleType'])  
y = df_lencoder['SaleType'] 

Untuk membagi dataset menjadi train set dan test set, kita bisa menggunakan fungsi train_test_split. Fungsi ini memerlukan beberapa parameter utama:

X: kumpulan atribut atau fitur dari dataset, yaitu semua kolom kecuali kolom target (pada kasus ini SalePrice).
y: target atau label yang akan kita prediksi merupakan kolom yang ingin kita pisahkan (pada kasus ini SalePrice).
test_size: persentase dari data yang akan digunakan sebagai test set. Misalnya, ketika Anda menentukan test_size=0.2, 20% dari data akan menjadi test set, dan 80% sisanya akan menjadi train set.
random_state: parameter yang memastikan pemisahan data yang konsisten setiap kali fungsi tersebut dijalankan. Parameter ini mengontrol pengacakan saat membagi dataset sehingga dengan menetapkan random_state hasil pembagian data (train set dan test set) akan selalu sama. Hal ini penting untuk memastikan eksperimen yang dapat direproduksi dan evaluasi model yang konsisten. Namun, parameter ini juga dapat memiliki kekurangan yaitu model tidak akan menerima inputan baru yang mungkin akan lebih baik dari sebelumnya.
Ketika Anda menjalankan train_test_split, fungsi ini akan mengembalikan empat bagian data seperti berikut.

X_train: fitur untuk train set.
X_test: fitur untuk test set.
y_train: target untuk train set.
y_test: target untuk test set.
Ini berarti setelah pemanggilan fungsi, Anda memiliki dua bagian data: satu untuk melatih model (train set) dan satu lagi untuk menguji model (test set).

In [ ]:
# membagi dataset menjadi training dan testing 
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
# menghitung panjang/jumlah data 
print("Jumlah data: ",len(X))
# menghitung panjang/jumlah data pada x_train
print("Jumlah data latih: ",len(x_train))
# menghitung panjang/jumlah data pada x_test
print("Jumlah data test: ",len(x_test))

**MODELLING**

In [ ]:
lars = linear_model.Lars(n_nonzero_coefs=1).fit(x_train, y_train)
 
# Melatih model 2 dengan algoritma Linear Regression
from sklearn.linear_model import LinearRegression
LR = LinearRegression().fit(x_train, y_train)
 
# Melatih model 3 dengan algoritma Gradient Boosting Regressor
from sklearn.ensemble import GradientBoostingRegressor
GBR = GradientBoostingRegressor(random_state=184)
GBR.fit(x_train, y_train)